# Judul PI :
```
Kerangka Prediksi Hoaks Multi-Layer untuk Konten Digital Indonesia: Integrasi Fitur Berbasis Transformer dan Verifikasi Sumber Eksternal
```
### Scraping Dataset
### Scrape Kompas

In [ ]:
import requests
import re
import pandas as pd
from typing import List, Dict

BASE_URL = "https://www.kompas.com/cekfakta/data-dan-fakta/{}"
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}


def fetch_html(url: str) -> str:
    """Fetch HTML content from a URL."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        return response.text
    except requests.RequestException:
        return ""


def extract_article_links(html: str) -> List[str]:
    """Extract valid article links from page HTML."""
    links = re.findall(r'href="(.*?)"', html)
    return [link for link in links if "cekfakta/read" in link]


def scrape_links(pages: int = 60) -> List[str]:
    """Scrape all article links from multiple pages."""
    all_links = set()

    for page in range(1, pages + 1):
        url = BASE_URL.format(page)
        html = fetch_html(url)
        if not html:
            continue

        links = extract_article_links(html)
        all_links.update(links)

    return list(all_links)


def parse_article(html: str) -> Dict[str, str]:
    """Extract title and content from article HTML."""
    # Title
    title_match = re.search(r'<h1 class="read__title">(.*?)</h1>', html)
    title = title_match.group(1).strip() if title_match else None

    # Content
    content_match = re.search(
        r'<div\s+[^>]*class=["\']read__content["\'][^>]*>(.*?)</div>',
        html,
        re.DOTALL
    )

    if not content_match:
        return {}

    raw_html = content_match.group(1)

    paragraphs = re.findall(r'<p[^>]*>(.*?)</p>', raw_html, re.DOTALL)
    clean_text = " ".join(paragraphs)
    clean_text = re.sub(r'<[^>]+>', '', clean_text).strip()

    if not title or not clean_text:
        return {}

    return {
        "TITLE": title,
        "NARRATIVE": clean_text,
        "LABEL": 1
    }


def scrape_articles(links: List[str]) -> List[Dict]:
    """Scrape all articles from given links."""
    data = []

    for link in links:
        html = fetch_html(link)
        if not html:
            continue

        article = parse_article(html)
        if article:
            data.append(article)

    return data


def save_to_csv(data: List[Dict], filename: str = "kompas_dataset.csv"):
    """Save scraped data to CSV."""
    df = pd.DataFrame(data)
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"Dataset saved to {filename}")


if __name__ == "__main__":
    links = scrape_links(pages=60)
    print(f"Collected {len(links)} unique links.")

    articles = scrape_articles(links)
    print(f"Collected {len(articles)} articles.")

    save_to_csv(articles)

Collected 1080 unique links.
Collected 1080 articles.
Dataset saved to kompas_dataset.csv


## Scrape Komdigi

### Fact News

In [ ]:
import requests
import json
import re
import pandas as pd
from typing import List, Dict

BASE_URL = "https://web.komdigi.go.id/api/v1/contents/category/berita-pemerintahan"
DETAIL_URL = "https://web.komdigi.go.id/api/v1/content/detail/berita-pemerintahan/{}"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "*/*",
    "Referer": "https://www.komdigi.go.id/",
    "Origin": "https://www.komdigi.go.id"
}


def fetch_json(url: str) -> Dict:
    """Fetch JSON data safely."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.RequestException:
        return {}


def get_slugs(pages: int = 10) -> List[str]:
    """Collect slugs from multiple pages."""
    all_slugs = set()

    for page in range(1, pages + 1):
        url = f"{BASE_URL}?perPage=12&page={page}&keyword="
        data = fetch_json(url)

        items = data.get("response", {}).get("data", [])
        for item in items:
            slug = item.get("slug")
            if slug:
                all_slugs.add(slug)

        print(f"Page {page}: collected {len(items)} items")

    return list(all_slugs)


def clean_html(text: str) -> str:
    """Remove HTML tags."""
    return re.sub(r"<[^>]+>", "", text).strip()


def fetch_article(slug: str) -> Dict:
    """Fetch single article detail."""
    url = DETAIL_URL.format(slug)
    data = fetch_json(url)

    article = data.get("data", {})
    title = article.get("title")
    body_html = article.get("body", "")

    clean_text = clean_html(body_html)

    if title and clean_text:
        return {
            "TITLE": title,
            "NARRATIVE": clean_text,
            "LABEL": 1
        }

    return {}


def scrape_articles(slugs: List[str]) -> List[Dict]:
    """Fetch all articles from slug list."""
    results = []

    for i, slug in enumerate(slugs, 1):
        article = fetch_article(slug)
        if article:
            results.append(article)

        if i % 10 == 0:
            print(f"Processed {i}/{len(slugs)} articles")

    return results


def save_to_csv(data: List[Dict], filename: str):
    """Save data to CSV."""
    df = pd.DataFrame(data)
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"Saved {len(df)} articles to {filename}")


if __name__ == "__main__":
    print("Collecting slugs...")
    slugs = get_slugs(pages=10)
    print(f"Total unique slugs: {len(slugs)}")

    print("\nFetching article details...")
    articles = scrape_articles(slugs)

    save_to_csv(articles, "komdigi_fact_dataset.csv")

    # Preview
    df_preview = pd.DataFrame(articles)
    print("\nPreview:")
    print(df_preview.head())

Page 1: collected 12 items
Page 2: collected 12 items
Page 3: collected 12 items
Page 4: collected 12 items
Page 5: collected 12 items
Page 6: collected 12 items
Page 7: collected 12 items
Page 8: collected 12 items
Page 9: collected 12 items
Page 10: collected 12 items
Total unique slugs: 120

Fetching article details...
Processed 10/120 articles
Processed 20/120 articles
Processed 30/120 articles
Processed 40/120 articles
Processed 50/120 articles
Processed 60/120 articles
Processed 70/120 articles
Processed 80/120 articles
Processed 90/120 articles
Processed 100/120 articles
Processed 110/120 articles
Processed 120/120 articles
Saved 120 articles to komdigi_fact_dataset.csv

Preview:
                                               TITLE  \
0  Presiden Beri Penghormatan Terakhir untuk Tiga...   
1  Satelit SATRIA Percepat Pemulihan Komunikasi W...   
2  Presiden Prabowo Bertolak ke Sumatra Utara, Sa...   
3  Indonesia–Australia Perkuat Kemitraan Strategi...   
4  Perkuat Koordinasi Pa

### Fake News

In [ ]:
import requests
import json
import re
import pandas as pd
from typing import List, Dict

BASE_URL = "https://web.komdigi.go.id/api/v1/contents/category/berita-hoaks"
DETAIL_URL = "https://web.komdigi.go.id/api/v1/content/detail/berita-hoaks/{}"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "*/*",
    "Referer": "https://www.komdigi.go.id/",
    "Origin": "https://www.komdigi.go.id"
}


def fetch_json(url: str) -> Dict:
    """Fetch JSON data safely."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.RequestException:
        return {}


def get_slugs(pages: int = 10) -> List[str]:
    """Collect slugs from multiple pages."""
    all_slugs = set()

    for page in range(1, pages + 1):
        url = f"{BASE_URL}?perPage=12&page={page}&keyword="
        data = fetch_json(url)

        items = data.get("response", {}).get("data", [])
        for item in items:
            slug = item.get("slug")
            if slug:
                all_slugs.add(slug)

        print(f"Page {page}: collected {len(items)} items")

    return list(all_slugs)


def clean_html(text: str) -> str:
    """Remove HTML tags."""
    return re.sub(r"<[^>]+>", "", text).strip()


def fetch_article(slug: str) -> Dict:
    """Fetch single article detail."""
    url = DETAIL_URL.format(slug)
    data = fetch_json(url)

    article = data.get("data", {})
    title = article.get("title")
    body_html = article.get("body", "")

    clean_text = clean_html(body_html)

    if title and clean_text:
        return {
            "TITLE": title,
            "NARRATIVE": clean_text,
            "LABEL": 0
        }

    return {}


def scrape_articles(slugs: List[str]) -> List[Dict]:
    """Fetch all articles from slug list."""
    results = []

    for i, slug in enumerate(slugs, 1):
        article = fetch_article(slug)
        if article:
            results.append(article)

        if i % 10 == 0:
            print(f"Processed {i}/{len(slugs)} articles")

    return results


def save_to_csv(data: List[Dict], filename: str):
    """Save data to CSV."""
    df = pd.DataFrame(data)
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"Saved {len(df)} articles to {filename}")


if __name__ == "__main__":
    print("Collecting slugs...")
    slugs = get_slugs(pages=10)
    print(f"Total unique slugs: {len(slugs)}")

    print("\nFetching article details...")
    articles = scrape_articles(slugs)

    save_to_csv(articles, "komdigi_hoax_dataset.csv")

    # Preview
    df_preview = pd.DataFrame(articles)
    print("\nPreview:")
    print(df_preview.head())

Page 1: collected 12 items
Page 2: collected 12 items
Page 3: collected 12 items
Page 4: collected 12 items
Page 5: collected 12 items
Page 6: collected 12 items
Page 7: collected 12 items
Page 8: collected 12 items
Page 9: collected 12 items
Page 10: collected 12 items
Total unique slugs: 120

Fetching article details...
Processed 10/120 articles
Processed 20/120 articles
Processed 30/120 articles
Processed 40/120 articles
Processed 50/120 articles
Processed 60/120 articles
Processed 70/120 articles
Processed 80/120 articles
Processed 90/120 articles
Processed 100/120 articles
Processed 110/120 articles
Processed 120/120 articles
Saved 120 articles to komdigi_hoax_dataset.csv

Preview:
                                               TITLE  \
0  [HOAKS] Menteri Luar Negeri Iran Abbas Araghch...   
1  [HOAKS] Lansia di atas 60 Tahun Tidak Bisa Tar...   
2  [HOAKS] Sri Mulyani Beri Bansos dengan Syarat ...   
3  [HOAKS] Akun Facebook dan WhatsApp Mengatasnam...   
4  [HOAKS] Akun WhatsApp

## Scrape TurnBackHoax

In [ ]:
import requests
import re
import pandas as pd
import concurrent.futures
from typing import List, Dict

BASE_URL = "https://turnbackhoax.id/articles?category=all&page={}"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}


def fetch_html(url: str) -> str:
    """Fetch HTML safely."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        return response.text
    except requests.RequestException:
        return ""


def collect_links(max_pages: int = 50) -> List[str]:
    """Collect article links from multiple pages."""
    all_links = set()

    for page in range(1, max_pages + 1):
        url = BASE_URL.format(page)
        html = fetch_html(url)

        links = re.findall(
            r'<a href="(.*?)" class="flex flex-col gap-1 lg:gap-2">',
            html
        )

        if not links:
            print(f"No more links at page {page}, stopping...")
            break

        all_links.update(links)
        print(f"Page {page}: +{len(links)} links (Total: {len(all_links)})")

    return list(all_links)


def clean_html(text: str) -> str:
    """Remove HTML tags."""
    return re.sub(r"<[^>]+>", "", text).strip()


def scrape_single_article(link: str) -> Dict:
    """Scrape a single article."""
    html = fetch_html(link)
    if not html:
        return {}

    try:
        title_match = re.search(
            r'<h1 class="text-3xl md:text-4xl md:leading-none">(.*?)</h1>',
            html
        )

        narasi_matches = re.findall(
            r'<p[^>]*>(.*?)</p>',
            html,
            re.DOTALL
        )

        title = title_match.group(1).strip() if title_match else None
        narasi_raw = " ".join(narasi_matches)
        narasi = clean_html(narasi_raw)

        if title and narasi:
            return {
                "TITLE": title,
                "NARRATIVE": narasi,
                "LABEL": 0  # 0 for hoax dataset
            }

    except Exception as e:
        print(f"Error parsing {link}: {e}")

    return {}


def scrape_articles_threaded(links: List[str], workers: int = 15) -> List[Dict]:
    """Scrape articles using multithreading."""
    results = []

    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as executor:
        futures = [executor.submit(scrape_single_article, link) for link in links]

        for i, future in enumerate(concurrent.futures.as_completed(futures), 1):
            data = future.result()
            if data:
                results.append(data)

            if i % 20 == 0:
                print(f"Processed {i}/{len(links)} articles")

    return results


def save_to_csv(data: List[Dict], filename: str):
    """Save dataset to CSV."""
    df = pd.DataFrame(data)
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"Saved {len(df)} articles to {filename}")


if __name__ == "__main__":
    print("Collecting links...")
    links = collect_links(max_pages=25)
    print(f"Total unique links: {len(links)}")

    print("\nStarting threaded scraping...")
    articles = scrape_articles_threaded(links, workers=15)

    save_to_csv(articles, "turnbackhoax_dataset.csv")

    # Preview
    df_preview = pd.DataFrame(articles)
    print("\nPreview:")
    print(df_preview.head())

Page 1: +10 links (Total: 10)
Page 2: +10 links (Total: 20)
Page 3: +10 links (Total: 30)
Page 4: +10 links (Total: 40)
Page 5: +10 links (Total: 50)
Page 6: +10 links (Total: 60)
Page 7: +10 links (Total: 70)
Page 8: +10 links (Total: 80)
Page 9: +10 links (Total: 90)
Page 10: +10 links (Total: 100)
Page 11: +10 links (Total: 110)
Page 12: +10 links (Total: 120)
Page 13: +10 links (Total: 130)
Page 14: +10 links (Total: 140)
Page 15: +10 links (Total: 150)
Page 16: +10 links (Total: 160)
Page 17: +10 links (Total: 170)
Page 18: +10 links (Total: 180)
Page 19: +10 links (Total: 190)
Page 20: +10 links (Total: 200)
Page 21: +10 links (Total: 210)
Page 22: +10 links (Total: 220)
Page 23: +10 links (Total: 230)
Page 24: +10 links (Total: 240)
Page 25: +10 links (Total: 250)
Total unique links: 250

Starting threaded scraping...
Processed 20/250 articles
Processed 40/250 articles
Processed 60/250 articles
Processed 80/250 articles
Processed 100/250 articles
Processed 120/250 articles
Proce

## Merge Dataset To One File

In [ ]:
import pandas as pd
from typing import List

FILE_PATHS = [
    'scraped_hoax_articles_threaded.csv',
    'kompas_dataset.csv',
    'komdigi_fact_dataset.csv',
    'komdigi_hoax_dataset.csv',
    'turnbackhoax_dataset.csv'
]


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize column names to TITLE, NARRATIVE, LABEL."""
    df.columns = [col.upper() for col in df.columns]

    # Handle possible variations
    column_mapping = {
        "TITLE": "TITLE",
        "NARASI": "NARRATIVE",
        "CONTENT": "NARRATIVE",
        "TEXT": "NARRATIVE",
        "LABEL": "LABEL"
    }

    df = df.rename(columns=column_mapping)

    # Check required columns
    required_cols = {"TITLE", "NARRATIVE", "LABEL"}
    if not required_cols.issubset(df.columns):
        return pd.DataFrame()  # return empty if invalid

    return df[list(required_cols)]


def load_datasets(paths: List[str]) -> List[pd.DataFrame]:
    """Load and clean multiple datasets."""
    dfs = []

    for path in paths:
        try:
            df = pd.read_csv(path)

            df = standardize_columns(df)
            if df.empty:
                print(f"Skipping {path} (invalid format)")
                continue

            # Drop empty rows
            df = df.dropna(subset=["TITLE", "NARRATIVE"])

            # Normalize label (ensure int)
            df["LABEL"] = df["LABEL"].astype(int)

            dfs.append(df)
            print(f"Loaded {len(df)} rows from {path}")

        except FileNotFoundError:
            print(f"File not found: {path}")
        except Exception as e:
            print(f"Error loading {path}: {e}")

    return dfs


def merge_datasets(dfs: List[pd.DataFrame]) -> pd.DataFrame:
    """Merge and clean datasets."""
    merged = pd.concat(dfs, ignore_index=True)

    # Remove duplicates
    merged = merged.drop_duplicates(subset=["TITLE", "NARRATIVE"])

    return merged


def save_dataset(df: pd.DataFrame, filename: str = "dataset.csv"):
    """Save final dataset."""
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"\nSaved {len(df)} rows to {filename}")


if __name__ == "__main__":
    print("Loading datasets...")
    dfs = load_datasets(FILE_PATHS)

    if not dfs:
        print("No valid datasets found.")
        exit()

    print("\nMerging datasets...")
    merged_df = merge_datasets(dfs)

    save_dataset(merged_df, "dataset.csv")

    # Preview
    print("\nPreview:")
    print(merged_df.head())

    print("\nDataset info:")
    print(merged_df.info())

    # Optional: label distribution
    print("\nLabel distribution:")
    print(merged_df["LABEL"].value_counts())

Loading datasets...
File not found: scraped_hoax_articles_threaded.csv
Loaded 1080 rows from kompas_dataset.csv
Loaded 120 rows from komdigi_fact_dataset.csv
Loaded 120 rows from komdigi_hoax_dataset.csv
Loaded 250 rows from turnbackhoax_dataset.csv

Merging datasets...

Saved 1570 rows to dataset.csv

Preview:
                                           NARRATIVE  \
0  KOMPAS.com - Mata dunia tertuju ke Rafah, kota...   
1  KOMPAS.com -&nbsp;Inisiasi untuk mengawal Pemi...   
2  KOMPAS.com - Komisi Pemilihan Umum (KPU) telah...   
3  KOMPAS.com - Sebanyak Rp 943 triliun transaksi...   
4  KOMPAS.com - Pemerintahan Presiden Joko Widodo...   

                                               TITLE  LABEL  
0  Fakta Serangan Israel ke Rafah, Kamp Pengungsi...      1  
1  Inisiatif Mengawal Suara Pemilu yang Dilakukan...      1  
2  INFOGRAFIK: Jumlah dan Persebaran Pemilih di S...      1  
3  PPATK Periksa Rp 934 Triliun Transaksi Perpaja...      1  
4  INFOGRAFIK: Bagaimana Tingkat Inflasi

# Hoax Classification Pipeline: IndoBERT + Machine Learning

This notebook implements a professional Machine Learning pipeline for Indonesian hoax classification. We combine state-of-the-art transformer-based feature extraction (**IndoBERT**) with robust traditional classifiers (**SVM**, **Random Forest**, and **Naive Bayes**).

### Project Workflow
1. **Data Cleaning**: Duplicate removal and null handling.
2. **NLP Preprocessing**: Case folding, stopword removal, and stemming using the `Sastrawi` library.
3. **Feature Engineering**: Extracting semantic embeddings using the `indobenchmark/indobert-base-p1` model.
4. **Modeling**: Training and evaluating multiple classifiers to compare performance.
5. **Evaluation**: Detailed metrics including Accuracy, F1-Score, and Precision-Recall.

## 1. Setup & Dependencies

In [27]:
!pip install pandas scikit-learn nltk Sastrawi transformers torch tqdm openai

In [28]:
import pandas as pd
import numpy as np
import re
import nltk
import torch
import os
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from transformers import AutoTokenizer, AutoModel
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Ensure NLTK data is available
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [29]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import nltk
import re

class HoaxPreprocessor:
    def __init__(self):
        print("Initializing Sastrawi components...")
        self.stemmer = StemmerFactory().create_stemmer()
        self.stopword_remover = StopWordRemoverFactory().get_stop_words()

    def clean_text(self, text):
        if not isinstance(text, str):
            return ""

        # Case folding
        text = text.lower()
        # Remove URLs, punctuations, and numbers
        text = re.sub(r'http\S+|www.\S+', '', text)
        text = re.sub(r'[^a-z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()

        # Tokenization & Stopword Removal
        tokens = nltk.tokenize.word_tokenize(text)
        tokens = [word for word in tokens if word not in self.stopword_remover]

        # Stemming
        cleaned_text = ' '.join(tokens)
        return self.stemmer.stem(cleaned_text)

    def process_dataframe(self, df, sample_size=None):
        # Drop duplicates and NAs
        df = df.drop_duplicates(subset=['NARRATIVE', 'TITLE']).dropna(subset=['NARRATIVE', 'TITLE', 'LABEL'])

        if sample_size:
            df = df.sample(n=min(sample_size, len(df)), random_state=42).copy()
        else:
            df = df.copy()

        tqdm.pandas(desc="Cleaning Text")
        print("Starting text preprocessing...")
        df['clean_combined'] = (df['TITLE'] + " " + df['NARRATIVE']).progress_apply(self.clean_text)
        return df

In [30]:
def get_embeddings(texts, model_name="indobenchmark/indobert-base-p1", batch_size=16):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    # Select Hardware
    device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
    model.to(device)
    model.eval()

    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting Features"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # Mean Pooling
        mask = inputs['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
        sum_embeddings = torch.sum(outputs.last_hidden_state * mask, 1)
        sum_mask = torch.clamp(mask.sum(1), min=1e-9)
        mean_pooled = (sum_embeddings / sum_mask).cpu().numpy()
        embeddings.append(mean_pooled)

    return np.vstack(embeddings)

In [ ]:
DATA_PATH = 'dataset.csv'
SAMPLE_SIZE = None # Set to None for full dataset

if not os.path.exists(DATA_PATH):
    print(f"Dataset not found! Please ensure '{DATA_PATH}' is in the current directory.")
else:
    # 1. Load
    df_raw = pd.read_csv(DATA_PATH)

    # 2. Preprocess
    preprocessor = HoaxPreprocessor()
    df_clean = preprocessor.process_dataframe(df_raw, sample_size=SAMPLE_SIZE)

    # 3. Features
    X = get_embeddings(df_clean['clean_combined'])
    y = df_clean['LABEL'].values

    # 4. Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"\nData split complete. Training on {len(X_train)} samples, testing on {len(X_test)} samples.")

Initializing Sastrawi components...
Starting text preprocessing...


Cleaning Text:   0%|          | 0/1569 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Extracting Features:   0%|          | 0/99 [00:00<?, ?it/s]


Data split complete. Training on 1255 samples, testing on 314 samples.


In [ ]:
models = {
    "SVM (Linear)": SVC(kernel='linear', probability=True),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Naive Bayes": GaussianNB()
}

for name, model in models.items():
    print(f"\n--- Training {name} ---")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"F1-Score (Weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")
    print(classification_report(y_test, y_pred))


--- Training SVM (Linear) ---
Accuracy: 0.9904
F1-Score (Weighted): 0.9904
              precision    recall  f1-score   support

           0       1.00      0.96      0.98        68
           1       0.99      1.00      0.99       246

    accuracy                           0.99       314
   macro avg       0.99      0.98      0.99       314
weighted avg       0.99      0.99      0.99       314


--- Training Random Forest ---
Accuracy: 0.9459
F1-Score (Weighted): 0.9429
              precision    recall  f1-score   support

           0       1.00      0.75      0.86        68
           1       0.94      1.00      0.97       246

    accuracy                           0.95       314
   macro avg       0.97      0.88      0.91       314
weighted avg       0.95      0.95      0.94       314


--- Training Naive Bayes ---
Accuracy: 0.9140
F1-Score (Weighted): 0.9056
              precision    recall  f1-score   support

           0       1.00      0.60      0.75        68
         

In [ ]:
import joblib

# Save the best performing model (e.g., SVM)
svm_model = models["SVM (Linear)"]
joblib.dump(svm_model, 'svm_model.pkl')
print("SVM model saved as svm_model.pkl")

# You can also save other models if needed
joblib.dump(models["Random Forest"], 'random_forest_model.pkl')
joblib.dump(models["Naive Bayes"], 'naive_bayes_model.pkl')

SVM model saved as svm_model.pkl


['naive_bayes_model.pkl']

In [31]:
def predict_hoax(text: str, models: dict, preprocessor: HoaxPreprocessor, tokenizer, model_bert) -> dict:
    """Predicts if a given text is a hoax using all trained models."""
    clean_text_input = preprocessor.clean_text(text)

    # Get embedding for the input text
    device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
    model_bert.to(device)
    model_bert.eval()

    inputs = tokenizer([clean_text_input], padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)

    with torch.no_grad():
        outputs = model_bert(**inputs)

    mask = inputs['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
    sum_embeddings = torch.sum(outputs.last_hidden_state * mask, 1)
    sum_mask = torch.clamp(mask.sum(1), min=1e-9)
    embedding = (sum_embeddings / sum_mask).cpu().numpy()

    predictions = {}
    for name, ml_model in models.items():
        pred = ml_model.predict(embedding)[0]
        predictions[name] = "Hoax" if pred == 0 else "Fact"

    return predictions

In [32]:
from transformers import AutoTokenizer, AutoModel

# Load tokenizer and model for embeddings once
bert_model_name = "indobenchmark/indobert-base-p1"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModel.from_pretrained(bert_model_name)

# Instantiate the preprocessor (if not already done)
if 'preprocessor' not in locals():
    preprocessor = HoaxPreprocessor()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [33]:
import joblib
loaded_svm_model = joblib.load('svm_model.pkl')
print("SVM model loaded successfully.")

SVM model loaded successfully.


In [34]:
test_models = {"Loaded SVM": loaded_svm_model}
sample_text_for_test = "Pemerintah akan membagikan bantuan sosial tunai sebesar 1 juta rupiah kepada seluruh warga." # Replace with your test text
print(f"\nPredicting for text: '{sample_text_for_test}'")
predictions_with_loaded_model = predict_hoax(sample_text_for_test, test_models, preprocessor, bert_tokenizer, bert_model)

for model_name, prediction in predictions_with_loaded_model.items():
    print(f"  {model_name}: {prediction}")


Predicting for text: 'Pemerintah akan membagikan bantuan sosial tunai sebesar 1 juta rupiah kepada seluruh warga.'
  Loaded SVM: Fact


# Fetch Sample From X.com

In [35]:
import requests
import re

LINK_BERITA = "https://x.com/liputan6dotcom/status/2044046919195078828"
response = requests.get(LINK_BERITA).text
narative_sample = re.findall(r'"full_text":"(.*?)","id_str"', response)[0]
print (narative_sample)

Bripda AS resmi jadi tersangka atas kematian rekan polisi yang dianiaya di barak Polda Kepri. Kasus dipicu kekalahan main bola, berakhir di meja hukum. #Polri #KepulauanRiau https://t.co/CtqYsOBYgs


### Sample Hoax and Fact Articles for Testing

In [36]:
import joblib
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

# Load all trained models
models = {
    "SVM (Linear)": joblib.load('svm_model.pkl'),
    "Random Forest": joblib.load('random_forest_model.pkl'),
    "Naive Bayes": joblib.load('naive_bayes_model.pkl')
}

test_narative_sample = predict_hoax(narative_sample, models, preprocessor, bert_tokenizer, bert_model)
for model_name, prediction in test_narative_sample.items():
    print(f"  {model_name}: {prediction}")

  SVM (Linear): Fact
  Random Forest: Fact
  Naive Bayes: Fact


In [ ]:
import openai
import os

openai_api_key = ""

if openai_api_key:
    openai.api_key = openai_api_key
    print("OpenAI API key loaded.")
else:
    print("Warning: OpenAI API key not found. Please set it in Colab secrets or environment variables.")

OpenAI API key loaded.


In [38]:
def predict_hoax_openai(text: str) -> dict:
    """Predicts if a given text is a hoax using OpenAI's GPT model and requests relevant links."""
    if not openai.api_key:
        return {"error": "OpenAI API key is not set."}

    prompt_template = f"""Anda adalah asisten deteksi hoaks. Berikan klasifikasi apakah teks berita berikut adalah 'Hoaks' atau 'Fakta'. Jika 'Fakta', berikan juga setidaknya dua tautan artikel berita atau sumber yang relevan dan terpercaya untuk mendukung klaim tersebut. Jika 'Hoaks', berikan juga penjelasan singkat mengapa itu hoaks. Format jawaban Anda seperti ini:

Klasifikasi: [Hoaks/Fakta]
Penjelasan: [Penjelasan singkat]
Tautan Pendukung: [Daftar tautan, jika ada]

Berikut teks beritanya:
\"\"\"{text}\"\"\""""

    try:
        response = openai.chat.completions.create(
            model="gpt-3.5-turbo",  # Using a valid model name
            messages=[
                {"role": "system", "content": "You are a helpful and accurate hoax detection assistant."},
                {"role": "user", "content": prompt_template.format(text=text)}
            ],
            temperature=0.2, # Lower temperature for more focused and less speculative output
            max_completion_tokens=500 # Changed from max_tokens to max_completion_tokens
        )
        return response.choices[0].message.content
    except Exception as e:
        return {"error": f"Error calling OpenAI API: {e}"}

In [39]:
import re
import requests
from urllib.parse import urlparse

DOMAIN_METRICS_API_BASE = "https://bot11-q8bt.onrender.com/check-metrics?domain="

def extract_domains_from_text(text: str) -> list[str]:
    """Extracts unique domains from a given text."""
    # Regex to find URLs
    urls = re.findall(r'https?://(?:www\.)?([^/\s]+)', text)
    domains = set()
    for url in urls:
        # urlparse might include port numbers or paths, we only want the netloc (domain)
        parsed_url = urlparse(f"https://{url}") # Prepend https to make urlparse work correctly
        domain = parsed_url.netloc
        if domain:
            # Remove 'www.' prefix if present
            if domain.startswith('www.'):
                domain = domain[4:]
            domains.add(domain)
    return list(domains)

def get_domain_metrics(domain: str) -> dict:
    """Fetches domain metrics from the external API."""
    api_url = f"{DOMAIN_METRICS_API_BASE}{domain}"
    try:
        response = requests.get(api_url, timeout=10)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
        return response.json()
    except requests.RequestException as e:
        print(f"Error fetching metrics for {domain}: {e}")
        return {"domain": domain, "error": str(e)}

def get_domain_metrics_from_openai_result(openai_result_text: str) -> dict:
    """Extracts domains from OpenAI result and fetches their metrics."""
    # Look for 'Tautan Pendukung:' section
    links_section_match = re.search(r'Tautan Pendukung:\s*(.*)', openai_result_text, re.DOTALL)

    if not links_section_match:
        return {"message": "No 'Tautan Pendukung' section found in OpenAI result."}

    links_section = links_section_match.group(1)
    domains = extract_domains_from_text(links_section)

    if not domains:
        return {"message": "No domains found in 'Tautan Pendukung' section."}

    all_metrics = {}
    for domain in domains:
        metrics = get_domain_metrics(domain)
        all_metrics[domain] = metrics

    return all_metrics

In [40]:
print("--- Testing Narative Sample From X.COM ---")
openai_results_narative = predict_hoax_openai(narative_sample)
print(openai_results_narative)

--- Testing Narative Sample From X.COM ---
Klasifikasi: Fakta
Penjelasan: Berita ini adalah fakta bahwa Bripda AS resmi menjadi tersangka atas kematian rekan polisi yang dianiaya di barak Polda Kepri setelah kekalahan dalam permainan bola. Kasus ini telah diproses hukum.
Tautan Pendukung:
1. https://news.detik.com/berita/d-5791553/bripda-as-resmi-jadi-tersangka-kematian-rekan-polisi-yang-dianiaya-di-barrak-polda-kepri
2. https://www.cnnindonesia.com/nasional/20211110134045-12-719231/bripda-as-jadi-tersangka-kematian-rekan-polisi-yang-dianiaya


In [41]:
import json

print("--- Domain Metrics for OpenAI Sample ---")
sample_matrics = get_domain_metrics_from_openai_result(openai_results_narative)
print(json.dumps(sample_matrics, indent=2))

--- Domain Metrics for OpenAI Sample ---
{
  "cnnindonesia.com": {
    "domain": "cnnindonesia.com",
    "Domain Authority": 73,
    "Domain Rating": 82,
    "Spam Score": "1%",
    "Page Authority": 62,
    "Site Traffic": 6951665
  },
  "news.detik.com": {
    "domain": "news.detik.com",
    "Domain Authority": 93,
    "Domain Rating": 89,
    "Spam Score": "4%",
    "Page Authority": 70,
    "Site Traffic": 4464434
  }
}


In [42]:
def summarize_credibility(domain_metrics: dict) -> str:
    """Summarizes the credibility of news sources based on domain metrics."""
    summary_parts = []
    if not domain_metrics or "message" in domain_metrics:
        return "Tidak ada tautan pendukung atau metrik domain yang ditemukan untuk disimpulkan."

    for domain, metrics in domain_metrics.items():
        if "error" in metrics:
            summary_parts.append(f"Domain '{domain}': Gagal mendapatkan metrik - {metrics['error']}")
            continue

        da = metrics.get("Domain Authority")
        dr = metrics.get("Domain Rating")
        spam_score_str = metrics.get("Spam Score")

        credibility = []

        if da is not None:
            if da >= 70:
                credibility.append("Otoritas Domain (DA) sangat tinggi.")
            elif da >= 40:
                credibility.append("Otoritas Domain (DA) tinggi.")
            else:
                credibility.append("Otoritas Domain (DA) sedang/rendah.")

        if dr is not None:
            if dr >= 70:
                credibility.append("Peringkat Domain (DR) sangat tinggi.")
            elif dr >= 40:
                credibility.append("Peringkat Domain (DR) tinggi.")
            else:
                credibility.append("Peringkat Domain (DR) sedang/rendah.")

        if spam_score_str:
            try:
                spam_score = int(spam_score_str.replace('%', ''))
                if spam_score <= 10:
                    credibility.append("Skor Spam rendah (indikasi kepercayaan).")
                elif spam_score <= 30:
                    credibility.append("Skor Spam sedang (perlu perhatian).")
                else:
                    credibility.append("Skor Spam tinggi (potensi tidak terpercaya).")
            except ValueError:
                credibility.append(f"Skor Spam tidak valid '{spam_score_str}'.")

        if credibility:
            summary_parts.append(f"Domain '{domain}': {' '.join(credibility)}")
        else:
            summary_parts.append(f"Domain '{domain}': Tidak ada metrik yang cukup untuk disimpulkan.")

    return "\n".join(summary_parts)

In [43]:
print("\n--- Ringkasan Kredibilitas Berita Sampel ---")
print(summarize_credibility(sample_matrics))


--- Ringkasan Kredibilitas Berita Sampel ---
Domain 'cnnindonesia.com': Otoritas Domain (DA) sangat tinggi. Peringkat Domain (DR) sangat tinggi. Skor Spam rendah (indikasi kepercayaan).
Domain 'news.detik.com': Otoritas Domain (DA) sangat tinggi. Peringkat Domain (DR) sangat tinggi. Skor Spam rendah (indikasi kepercayaan).
